In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_train_features = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_test_features = prepare_features(X_test)

y = train[TARGET_COLUMN].copy()

Используем почти тот же набор признаков, что и Ridge:

In [4]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS
]

X = X_train_features[feature_columns].copy()
X_final_test = X_test_features[feature_columns].copy()

assert list(X.columns) == list(X_final_test.columns)
assert "Предложение" not in X.columns
assert "car_id" not in X.columns

Определяем числовые и категориальные признаки:

In [5]:
numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

categorical_columns = [
    column
    for column in feature_columns
    if column not in numeric_columns
]

print("Числовых признаков:", len(numeric_columns))
print("Категориальных признаков:", len(categorical_columns))
print(categorical_columns)

Числовых признаков: 9
Категориальных признаков: 13
['Бренд', 'Модель', 'Тип машины', 'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод', 'Топливо', 'Цвет', 'Локация', 'Тип кузова', 'Штат']


Для CatBoost категориальные пропуски превращаем в отдельную строковую категорию:

In [6]:
for column in categorical_columns:
    X[column] = (
        X[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_final_test[column] = (
        X_final_test[column]
        .fillna("__MISSING__")
        .astype(str)
    )

Создаём тот же stratified split:

In [7]:
target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

print(X_train_split.shape)
print(X_valid_split.shape)

(6672, 22)
(1668, 22)


Обучаемся на mape

In [8]:
catboost_mape_model = CatBoostRegressor(
    loss_function="MAPE",
    eval_metric="MAPE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_mape_model.fit(
    X_train_split,
    y_train_split,
    cat_features=categorical_columns,
    eval_set=(
        X_valid_split,
        y_valid_split,
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.4985092	test: 0.5132100	best: 0.5132100 (0)	total: 356ms	remaining: 17m 47s
300:	learn: 0.3025138	test: 0.3387896	best: 0.3387896 (300)	total: 56.4s	remaining: 8m 25s
600:	learn: 0.2837014	test: 0.3292144	best: 0.3292054 (589)	total: 1m 58s	remaining: 7m 52s
900:	learn: 0.2737243	test: 0.3246869	best: 0.3246869 (900)	total: 2m 47s	remaining: 6m 30s
1200:	learn: 0.2661709	test: 0.3209251	best: 0.3208107 (1199)	total: 3m 47s	remaining: 5m 41s
1500:	learn: 0.2608216	test: 0.3188010	best: 0.3187992 (1499)	total: 4m 33s	remaining: 4m 33s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.318719397
bestIteration = 1508

Shrink model to first 1509 iterations.


CatBoostRegressor(allow_writing_files=False, depth=8, eval_metric='MAPE', iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='MAPE', random_seed=42, verbose=300)

In [11]:
from sklearn.metrics import mean_absolute_percentage_error


def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100

Предсказание

In [12]:
y_pred_train_mape = np.maximum(
    catboost_mape_model.predict(X_train_split),
    1,
)

y_pred_valid_mape = np.maximum(
    catboost_mape_model.predict(X_valid_split),
    1,
)

catboost_mape_result = pd.DataFrame(
    {
        "model": [
            "CatBoost MAPE target_original_scale"
        ],
        "best_iteration": [
            catboost_mape_model.get_best_iteration()
        ],
        "train_mape_pct": [
            round(
                mape_percent(y_train_split, y_pred_train_mape),
                3,
            )
        ],
        "validation_mape_pct": [
            round(
                mape_percent(y_valid_split, y_pred_valid_mape),
                3,
            )
        ],
    }
)

comparison = pd.DataFrame(
    {
        "model": [
            "CatBoost RMSE on log1p(target), 3000",
            "CatBoost MAPE on original target, 3000",
        ],
        "validation_mape_pct": [
            13.433,
            catboost_mape_result.loc[
                0, "validation_mape_pct"
            ],
        ],
    }
)

display(catboost_mape_result)
display(comparison)

,model,best_iteration,train_mape_pct,validation_mape_pct
0,CatBoost MAPE target_original_scale,1508,28.428,31.872


,model,validation_mape_pct
0,"CatBoost RMSE on log1p(target), 3000",13.433
1,"CatBoost MAPE on original target, 3000",31.872
